# LOF

The Local Outlier Factor (LOF) compares the density around each point with the density around its neighbours. A point in a sparse spot next to a dense cluster gets a high score, even if its absolute distance to the cluster is small.

In [ ]:
using OutlierDetection, OutlierDetectionNeighbors
using Plots, Statistics, Random

Random.seed!(2);

## Data with Two Densities

A tight cluster and a wide cluster, plus two points just outside the tight one. In absolute distance they are closer to their neighbours than many ordinary points of the wide cluster are.

In [ ]:
tight = 0.3 .* randn(150, 2)
wide  = 2.0 .* randn(150, 2) .+ [8.0 0.0]
local_out = [1.5 1.5; -1.5 1.4]
X = vcat(tight, wide, local_out)
is_true = vcat(falses(300), trues(2))

scatter(X[:, 1], X[:, 2]; group = ifelse.(is_true, "Local outlier", "Normal"),
    markerstrokewidth = 0, xlabel = "Feature 1", ylabel = "Feature 2", title = "Data")

## KNN Versus LOF

KNN uses one global distance scale, so the edge of the wide cluster dominates its top scores. LOF normalises by local density.

In [ ]:
_, knn_scores = OutlierDetection.fit(KNNDetector(k = 10), permutedims(X); verbosity = 0)
_, lof_scores = OutlierDetection.fit(LOFDetector(k = 10), permutedims(X); verbosity = 0)

rank_of(s, i) = count(>=(s[i]), s)   # 1 = highest score
for i in findall(is_true)
    println("Point $i  KNN rank: ", rank_of(knn_scores, i), "   LOF rank: ", rank_of(lof_scores, i))
end

In [ ]:
p1 = scatter(X[:, 1], X[:, 2]; zcolor = knn_scores, markerstrokewidth = 0, title = "KNN", label = "")
p2 = scatter(X[:, 1], X[:, 2]; zcolor = lof_scores, markerstrokewidth = 0, title = "LOF", label = "")
plot(p1, p2; layout = (1, 2), size = (900, 380))

## Reading LOF Scores

The score is near 1 for points as dense as their neighbours and grows as a point becomes relatively isolated. Values of 1.5 to 2 are usually notable; there is no universal cut-off, so flag the top share instead.

In [ ]:
threshold = quantile(lof_scores, 0.99)
println("Flagged: ", findall(lof_scores .> threshold))
println("Median LOF score: ", round(median(lof_scores), digits = 2))

## Sensitivity to k

k must exceed the size of any small group of outliers you want to catch; otherwise they form their own dense neighbourhood.

In [ ]:
for k in (5, 10, 20, 40)
    _, s = OutlierDetection.fit(LOFDetector(k = k), permutedims(X); verbosity = 0)
    println("k = ", rpad(k, 3), " ranks of local outliers: ", [rank_of(s, i) for i in findall(is_true)])
end

## Summary
- LOF compares a point's local density with its neighbours' densities.
- It finds local outliers next to dense clusters that KNN misses.
- Scores near 1 are normal; flag the top share rather than a fixed value.
- Check several values of k.